<a href="https://colab.research.google.com/github/Priyaa1904/Flyrank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Priyaa1904/Flyrank-ML-Internship-Starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of analysis + time window

For the Content Refresh lane, one row represents one content item for one client on one report date. I will use `fact_content_daily_performance` as the main table. For development and verification, I will work on the March 2026 partition rather than the final June 2026 month. The goal is to use information available up to the decision point to rank content items for review, while keeping future outcome information out of the feature set.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Fields

**Features**
- `gsc_impressions` — historical search impressions available before the decision.
- `gsc_avg_position` — historical search position available before the decision.
- `gsc_clicks` — historical search clicks available before the decision.
- `word_count` — stored content attribute available before the decision.
- `search_volume` — stored search-demand attribute available before the decision.

**Label / proxy**
- `is_declining_proxy` — the outcome/proxy used to identify declining content. It is an outcome, not a feature.

**Context**
- `client_hash_id` — identifies the client for grouping, joining, and validation; it is not a model feature.
- `content_hash_id` — identifies the content item; it is not a model feature.
- `report_date` — identifies the observation date.

**Excluded**
- Future outcome information is excluded because it would not be known at the March decision moment and could cause leakage.
- `trend_direction` and `trend_pct` are excluded because the declining label is derived from the trend signal; using them as features would leak label information.

In [32]:
%pip -q install duckdb

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("✅ DuckDB connected to Hugging Face")

✅ DuckDB connected to Hugging Face


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification

I verified the data contract on the March 2026 partition using three queries. First, I checked the expected grain by looking for duplicate combinations of report date, client, and content. Second, I measured the row count and date span of the March slice. Third, I checked analytics availability using the `ga4_data_available IS TRUE` flag.

The checks found 0 duplicate grain combinations, 9,841,378 rows from March 1 through March 31, 2026, and 413,966 rows where GA4 data is explicitly available.

In [33]:
REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM {REL}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Duplicate grain combinations found:", len(grain_check))
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain combinations found: 0


,report_date,client_hash_id,content_hash_id,row_count


In [34]:
march_summary = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM {REL}
""").df()

march_summary

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [35]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS rows_with_ga4_available
    FROM {REL}
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_with_ga4_available
0,9841378,413966


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limitation

A limitation of this slice is uneven analytics availability across clients and dates. In March 2026, only 413,966 of 9,841,378 rows have `ga4_data_available IS TRUE`. Therefore, GA4-based signals cannot be assumed to be available for every content item. This means a ranking system using those signals could have less information for some pages and the observed performance may not generalize equally across all clients.

In [36]:
content_rel = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
)
"""

feature_frame = con.sql(f"""
    SELECT
        p.report_date,
        p.client_hash_id,
        p.content_hash_id,

        p.gsc_impressions,
        p.gsc_avg_position,
        p.gsc_clicks,

        c.word_count,
        c.search_volume

    FROM {REL} AS p

    LEFT JOIN {content_rel} AS c
        ON p.client_hash_id = c.client_hash_id
        AND p.content_hash_id = c.content_hash_id

    WHERE p.gsc_data_available IS TRUE

    LIMIT 1000
""").df()

feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,gsc_clicks,word_count,search_volume
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,3.350000,0,<NA>,20
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0.000000,0,<NA>,10
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,4.928000,1,2123,20
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,4.000000,0,<NA>,90
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,2.272727,0,<NA>,40


### Five features and availability

I use five features for the initial feature frame:

1. **`gsc_impressions`** — available at the decision moment because it represents historical search impressions observed for the content item.
2. **`gsc_avg_position`** — available at the decision moment because it represents observed search position for the content item.
3. **`gsc_clicks`** — available at the decision moment because these are historical search clicks observed before prioritization.
4. **`word_count`** — available at the decision moment because it is a stored content attribute.
5. **`search_volume`** — available at the decision moment because it is a stored search-demand attribute associated with the content item.

These features describe information that could be available when deciding which content to prioritize. The future outcome or label is not used as an input feature.

In [37]:
APR_REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet')"

april_check = con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM {APR_REL}
""").df()

april_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,start_date,end_date
0,10424730,2026-04-01,2026-04-30


In [38]:
# Build a future-derived proxy for the leakage demonstration.
# March = decision-time information
# April = future outcome information

leak_demo = con.sql(f"""
    WITH march AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS march_impressions
        FROM {REL}
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),

    april AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS april_impressions
        FROM {APR_REL}
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )

    SELECT
        m.client_hash_id,
        m.content_hash_id,
        m.march_impressions,
        a.april_impressions,

        CASE
            WHEN a.april_impressions < m.march_impressions
            THEN 1
            ELSE 0
        END AS is_declining_proxy

    FROM march AS m
    INNER JOIN april AS a
        ON m.client_hash_id = a.client_hash_id
        AND m.content_hash_id = a.content_hash_id

    LIMIT 1000
""").df()

leak_demo.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,march_impressions,april_impressions,is_declining_proxy
0,client_62f4a7e64f5e0096,content_76c1f31e2b38f054,747.0,249.0,1
1,client_62f4a7e64f5e0096,content_ffc5ab4b34aab1f8,501.0,281.0,1
2,client_62f4a7e64f5e0096,content_9739856fc83dc1ca,2893.0,722.0,1
3,client_62f4a7e64f5e0096,content_3d1dc691a3502105,6955.0,3068.0,1
4,client_62f4a7e64f5e0096,content_9d28af4f99c5e67b,6789.0,3896.0,1


In [39]:
# Deliberate leakage experiment

leak_demo["leaked_feature"] = leak_demo["is_declining_proxy"]

# Quick leakage score:
# If the leaked feature is the label itself, it predicts the label perfectly.
leak_score = (
    leak_demo["leaked_feature"] == leak_demo["is_declining_proxy"]
).mean()

print(f"Deliberate leakage score: {leak_score:.3f}")

Deliberate leakage score: 1.000


### The leakage trap

I deliberately created `leaked_feature` by copying the outcome proxy itself. This produces a perfect quick score of 1.000 because the feature directly contains the information we are trying to predict.

This is invalid in a real prediction system: the outcome is not known at the March decision moment. The experiment demonstrates why label-derived information must never be included as a feature.

The leaked feature is removed immediately after the demonstration.

In [40]:
# Remove the deliberately leaked feature

leak_demo = leak_demo.drop(columns=["leaked_feature"])

print("✅ Leaked feature removed.")
print("Remaining columns:")
print(leak_demo.columns.tolist())

✅ Leaked feature removed.
Remaining columns:
['client_hash_id', 'content_hash_id', 'march_impressions', 'april_impressions', 'is_declining_proxy']


### Final leakage decision

The deliberately leaked feature has been removed. The final feature set does not use April outcome information. This keeps the features aligned with what would have been knowable at the March decision moment.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.